As part of your analysis, you'll conduct hypothesis testing to make data-driven conclusions about the effectiveness of the redesign. See the full details below:

Completion Rate
Completion Rate with a Cost-Effectiveness Threshold
Other Hypothesis Examples
Experiment Evaluation

You are also required to carry out an evaluation of the experiment by answering questions about the design effectiveness, duration and any additional data needs. See the full details below:

Design Effectiveness
Duration Assessment
Was the timeframe of the experiment (from 3/15/2017 to 6/20/2017) adequate to gather meaningful data and insights?

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

ROOT = Path().resolve().parent
DATA_PATH = ROOT / "data_raw"

print("Directorio actual:", ROOT)
print("Ruta de datos:", DATA_PATH)

Directorio actual: /Users/gabrielbohorquez/Desktop/Ironhack/Semana_5/PROYECTO_2/Projecto-2-Vanguard-ab-test
Ruta de datos: /Users/gabrielbohorquez/Desktop/Ironhack/Semana_5/PROYECTO_2/Projecto-2-Vanguard-ab-test/data_raw


In [19]:
# Cargar datos limpios
df_demo       = pd.read_csv(DATA_PATH / "df_demo_clean.csv")
df_experiment = pd.read_csv(DATA_PATH / "df_experiment_clean.csv")
df_web        = pd.read_csv(DATA_PATH / "df_web_clean.csv")

print("Datasets cargados correctamente!")
print("df_demo_clean:      ", len(df_demo), "filas")
print("df_experiment_clean:", len(df_experiment), "filas")
print("df_web_clean:       ", len(df_web), "filas")

Datasets cargados correctamente!
df_demo_clean:       70591 filas
df_experiment_clean: 50500 filas
df_web_clean:        744641 filas


In [20]:
# Cargar datos limpios
import pandas as pd

def inspect_datasets(dataframes: list[pd.DataFrame], names: list[str]) -> None:
    """
    Imprime las columnas de cada dataset para verificar su estructura.
    """
    for df, name in zip(dataframes, names):
        print(f"\n--- Columnas en {name} ---")
        print(list(df.columns))

# Lista de tus DataFrames y sus nombres para el bucle
datasets = [df_demo, df_experiment, df_web]
dataset_names = ["df_demo", "df_experiment", "df_web"]

# Ejecutar la inspección
inspect_datasets(datasets, dataset_names)




--- Columnas en df_demo ---
['client_id', 'clnt_tenure_yr', 'clnt_tenure_mnth', 'clnt_age', 'gendr', 'num_accts', 'bal', 'calls_6_mnth', 'logons_6_mnth']

--- Columnas en df_experiment ---
['client_id', 'Variation']

--- Columnas en df_web ---
['client_id', 'visitor_id', 'visit_id', 'process_step', 'date_time']


In [6]:
def check_content(dataframes: list[pd.DataFrame], names: list[str]) -> None:
    """
    Muestra las primeras filas de cada dataset de forma organizada.
    """
    for df, name in zip(dataframes, names):
        print(f"\n{'='*10} Contenido de {name} {'='*10}")
        # Usamos display(df.head()) si estás en Jupyter, o print() en script puro
        print(df.head(3)) 

# Ejecutar la revisión
datasets = [df_demo, df_experiment, df_web]
dataset_names = ["df_demo", "df_experiment", "df_web"]

check_content(datasets, dataset_names)


========== Contenido de df_demo ==========
   client_id  clnt_tenure_yr  clnt_tenure_mnth  clnt_age gendr  num_accts  \
0     836976             6.0              73.0      60.5     U        2.0   
1    2304905             7.0              94.0      58.0     U        2.0   
2    1439522             5.0              64.0      32.0     U        2.0   

         bal  calls_6_mnth  logons_6_mnth  
0   45105.30           6.0            9.0  
1  110860.30           6.0            9.0  
2   52467.79           6.0            9.0  

========== Contenido de df_experiment ==========
   client_id Variation
0    9988021      Test
1    8320017      Test
2    4033851   Control

========== Contenido de df_web ==========
   client_id            visitor_id                      visit_id process_step  \
0    9988021  580560515_7732621733  781255054_21935453173_531117       step_3   
1    9988021  580560515_7732621733  781255054_21935453173_531117       step_2   
2    9988021  580560515_7732621733  7812550

# 1 analysis de la efectividad del rediseño.

Usaremos df_experiment para separar a los usuarios en Test (rediseño) y Control (diseño viejo) y lo cruzaremss con df_web para ver quién completó el proceso.

A. Completion Rate (Tasa de Finalización)

Hipótesis Nula (H_0): No hay diferencia en la tasa de finalización entre el grupo de Test y el de Control.

Hipótesis Alternativa (H_1): El nuevo diseño (Test) tiene una tasa de finalización significativamente mayor.

Método: Realizaremos un test para proporciones. Si el p-value es menor a 0.05, rechazamos H_0 y confirmamos que el rediseño es efectivo.

In [7]:
# hipotisis testing: Tasa de Finalización
# Identificamos quién llegó al paso 'confirm'
df_web['is_completed'] = (df_web['process_step'] == 'confirm').astype(int)

# Agrupamos por cliente (1 si completó al menos una vez, 0 si nunca)
df_conversion = df_web.groupby('client_id')['is_completed'].max().reset_index()

# Unimos con el experimento
df_test_logic = pd.merge(df_conversion, df_experiment, on='client_id')

# Separamos los grupos
group_control = df_test_logic[df_test_logic['Variation'] == 'Control']['is_completed']
group_test = df_test_logic[df_test_logic['Variation'] == 'Test']['is_completed']

In [10]:
# Usamos el Z-test porque se trata de proporciones (tasa de finalización). 

count = [group_test.sum(), group_control.sum()]
nobs = [len(group_test), len(group_control)]

z_stat, p_value = proportions_ztest(
    count=count,
    nobs=nobs,
    alternative="larger"
)

print(f"Tasa de Finalización - Test: {group_test.mean():.2%}")
print(f"Tasa de Finalización - Control: {group_control.mean():.2%}")
print(f"Z-statistic: {z_stat:.4f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("La diferencia es estadísticamente significativa.")
else:
    print("No se observa diferencia significativa.")

Tasa de Finalización - Test: 69.29%
Tasa de Finalización - Control: 65.59%
Z-statistic: 8.8745
P-value: 0.0000
La diferencia es estadísticamente significativa.


Conclusiones
- Design Effectiveness: teniendo en cuenta que el p-value es menor a 0.05 (0.000), podemos concluir que el rediseño tiene un impacto real en el comportamiento del usuario.
- Completion Rate: la tasa de finalización ha aumentado de 3,7 puntos.
- Teniendo en cuenta que el experimento abarca desde marzo hasta junio de 2017, este T-test es muy robusto porque el tamaño de la muestra acumulado en esos meses reduce significativamente el error estándar.


B. Cost-Effectiveness Threshold

Aquí no solo buscamos que sea "mejor", sino que supere un umbral (ej. un aumento del 5%.
Análisis: Si el coste de implementar el rediseño es alto, la mejora observada debe justificar la inversión.

In [11]:
# hpotisis testing: Cost-Effectiveness Threshold
# Definimos el umbral (5%)

threshold = 0.05

# Calculamos las tasas de finalización (medias)
rate_control = group_control.mean()
rate_test = group_test.mean()

# Calculamos la mejora absoluta
observed_diff = rate_test - rate_control

print(f"Mejora observada: {observed_diff:.2%}")

Mejora observada: 3.71%


In [12]:
observed_diff = rate_test - rate_control
print(f"Mejora observada: {observed_diff:.2%}")
print(f"Umbral requerido: {threshold:.2%}")

if observed_diff >= threshold:
    print("El rediseño supera el umbral de rentabilidad.")
else:
    print(f"El rediseño NO supera el umbral: {observed_diff:.2%} < {threshold:.2%}")

Mejora observada: 3.71%
Umbral requerido: 5.00%
El rediseño NO supera el umbral: 3.71% < 5.00%


Conclusion
La mejora actual (3.7%) está por debajo del umbral del 5% que definimos. 

Esto es clave para nuestra evaluación: indica que aunque el diseño es mejor, pero quizás no lo suficiente para justificar una inversión muy alta.

C. Edad y Tenencia 

Dado que tenemos df_demo, podemos verificar si el rediseño funciona mejor para ciertos segmentos:

¿Influye la edad (clnt_age) en la probabilidad de completar el proceso?

¿Los clientes más antiguos (clnt_tenure_yr) son más resistentes al cambio que los nuevos?

In [13]:
# hpotisis testing: Edad y Tenencia 
# Unimos los datos de conversión (que ya calculamos) con df_demo y df_experiment
df_segmented = pd.merge(df_conversion, df_demo, on='client_id')
df_segmented = pd.merge(df_segmented, df_experiment, on='client_id')

# Filtramos por los grupos de interés
test_group = df_segmented[df_segmented['Variation'] == 'Test']
control_group = df_segmented[df_segmented['Variation'] == 'Control']

In [14]:
# Comparamos la edad de los que finalizaron vs los que no en el grupo Test
completed_age = test_group[test_group['is_completed'] == 1]['clnt_age']
not_completed_age = test_group[test_group['is_completed'] == 0]['clnt_age']

# T-test para ver si hay una diferencia significativa de edad en el éxito
t_stat_age, p_val_age = stats.ttest_ind(completed_age, not_completed_age, equal_var=False)

print(f"Edad media (Completado): {completed_age.mean():.1f}")
print(f"Edad media (No Completado): {not_completed_age.mean():.1f}")
print(f"P-value edad: {p_val_age:.4f}")

Edad media (Completado): 46.6
Edad media (No Completado): 48.5
P-value edad: 0.0000


In [18]:
# Analizamos la antigüedad en el grupo que falló en el rediseño (Test) 
# frente al grupo que falló en el diseño viejo (Control)
tenure_fail_test = test_group[test_group['is_completed'] == 0]['clnt_tenure_yr']
tenure_fail_control = control_group[control_group['is_completed'] == 0]['clnt_tenure_yr']

t_stat_tenure, p_val_tenure = stats.ttest_ind(tenure_fail_test, 
                              tenure_fail_control, equal_var=False)

print(f"Antigüedad media en abandonos (Test): {tenure_fail_test.mean():.1f} años")
print(f"Antigüedad media en abandonos (Control): {tenure_fail_control.mean():.1f} años")
print(f"P-value antigüedad: {p_val_tenure:.4f}")

if p_val_tenure < 0.05:
    print("La antigüedad SÍ influye significativamente en el abandono.")
else:
    print("La antigüedad NO influye significativamente en el abandono.")

Antigüedad media en abandonos (Test): 12.1 años
Antigüedad media en abandonos (Control): 12.2 años
P-value antigüedad: 0.3275
La antigüedad NO influye significativamente en el abandono.


Concluciones

- Teniendo en cuenta que el P_value es inferior a 0.05 (0.0000), podemos afirmar que la edad si juega un papel en el resulado.

- La antigüedad media del grupo que ha fallado en el resideño a aumentado de 0.1